In [2]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [3]:
load_dotenv()
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [4]:
class JokeState(TypedDict):
    
    topic: str
    joke : str
    explanation: str

In [5]:
def generate_joke(state: JokeState):
    
    prompt = f"generate a joke on topic {state['topic']}"
    response = llm.invoke(prompt).content
    
    return {'joke': response}

In [6]:
def generate_explanation(state: JokeState):
    
    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content
    
    return {'explanation' : response}

In [7]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)


graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [8]:
config1 = {"configurable" :{"thread_id": "1"}}
workflow.invoke({'topic': 'pizza'}, config = config1)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


{'topic': 'pizza',
 'joke': 'Why did the pizza apply for a job?\n\nBecause it wanted to make some **dough**!',
 'explanation': 'This joke is a classic **pun**! It plays on the double meaning of the word "dough."\n\nHere\'s the breakdown:\n\n1.  **"Dough" (meaning 1):** This is the literal ingredient that pizza is made from (flour, water, yeast mixed together). A pizza *has* dough.\n2.  **"Dough" (meaning 2):** This is a common slang term for **money**. When someone applies for a job, they want to "make money."\n\nSo, the humor comes from the unexpected twist:\n*   A pizza literally *has* dough (the food).\n*   It applies for a job because it wants to *make money* (which is also called "dough").\n\nThe joke cleverly uses the word "dough" to refer to both the pizza\'s primary ingredient and the financial reward of working.'}

In [9]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?\n\nBecause it wanted to make some **dough**!', 'explanation': 'This joke is a classic **pun**! It plays on the double meaning of the word "dough."\n\nHere\'s the breakdown:\n\n1.  **"Dough" (meaning 1):** This is the literal ingredient that pizza is made from (flour, water, yeast mixed together). A pizza *has* dough.\n2.  **"Dough" (meaning 2):** This is a common slang term for **money**. When someone applies for a job, they want to "make money."\n\nSo, the humor comes from the unexpected twist:\n*   A pizza literally *has* dough (the food).\n*   It applies for a job because it wants to *make money* (which is also called "dough").\n\nThe joke cleverly uses the word "dough" to refer to both the pizza\'s primary ingredient and the financial reward of working.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1cebf7-0ad5-60a0-8002-8282ff4ee023'}}, metadata={'so

In [10]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?\n\nBecause it wanted to make some **dough**!', 'explanation': 'This joke is a classic **pun**! It plays on the double meaning of the word "dough."\n\nHere\'s the breakdown:\n\n1.  **"Dough" (meaning 1):** This is the literal ingredient that pizza is made from (flour, water, yeast mixed together). A pizza *has* dough.\n2.  **"Dough" (meaning 2):** This is a common slang term for **money**. When someone applies for a job, they want to "make money."\n\nSo, the humor comes from the unexpected twist:\n*   A pizza literally *has* dough (the food).\n*   It applies for a job because it wants to *make money* (which is also called "dough").\n\nThe joke cleverly uses the word "dough" to refer to both the pizza\'s primary ingredient and the financial reward of working.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1cebf7-0ad5-60a0-8002-8282ff4ee023'}}, metadata={'s

In [14]:
config2 = {"configurable": {"thread_id": "2", "checkpoint_id": "1f1cebf7-0be9-6f6b-8000-d195ed9279e8"}}
workflow.invoke({'topic': 'shinchan'}, config=config2)

{'topic': 'shinchan',
 'joke': 'Why did Shin-chan bring a ladder to kindergarten?\n\nBecause he heard the teacher wanted to raise his grades, and he wanted to make sure she had a *butt-er* view of his efforts!',
 'explanation': 'This joke plays on a few elements, especially if you\'re familiar with the character Shin-chan:\n\n1.  **Literal Interpretation:** Shin-chan is a mischievous, often literal-minded kindergartener.\n    *   When the teacher says she wants to **"raise his grades,"** she means to **improve** them (make them higher in value/score).\n    *   Shin-chan, however, interprets "raise" literally, as in to **physically lift them higher** off the ground.\n\n2.  **The Ladder:** If you want to physically raise something or get a better view of something that\'s been raised, a ladder helps you go higher. So, in his literal mind, a ladder is a tool for "raising" things.\n\n3.  **The Pun and Shin-chan\'s Personality:**\n    *   The punchline uses the pun **"butt-er view,"** which

In [12]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'shinchan', 'joke': 'Why did Shin-chan get kicked out of the library?\n\nBecause he kept doing the "Elephant Dance" and shouting, "Ooh la la!"', 'explanation': 'This joke relies on understanding the character of Shin-chan (Shinnosuke Nohara) and the typical rules of a library.\n\nHere\'s the breakdown:\n\n1.  **Shin-chan\'s Character:**\n    *   **Mischievous and Disobedient:** Shin-chan is a five-year-old boy known for his cheeky, often inappropriate, and highly disruptive behavior. He loves to annoy adults and break rules.\n    *   **The "Elephant Dance" (Zou-san Dance):** This is one of Shin-chan\'s most famous and infamous gags. He drops his pants, wiggles his butt, and often exposes himself (specifically his "elephant," referring to his penis), while chanting "Zou-san, Zou-san" or making other silly noises. It\'s his ultimate act of public indecency and attention-seeking.\n    *   **"Ooh la la!":** While not his primary catchphrase, Shin-chan often u

In [13]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'shinchan', 'joke': 'Why did Shin-chan get kicked out of the library?\n\nBecause he kept doing the "Elephant Dance" and shouting, "Ooh la la!"', 'explanation': 'This joke relies on understanding the character of Shin-chan (Shinnosuke Nohara) and the typical rules of a library.\n\nHere\'s the breakdown:\n\n1.  **Shin-chan\'s Character:**\n    *   **Mischievous and Disobedient:** Shin-chan is a five-year-old boy known for his cheeky, often inappropriate, and highly disruptive behavior. He loves to annoy adults and break rules.\n    *   **The "Elephant Dance" (Zou-san Dance):** This is one of Shin-chan\'s most famous and infamous gags. He drops his pants, wiggles his butt, and often exposes himself (specifically his "elephant," referring to his penis), while chanting "Zou-san, Zou-san" or making other silly noises. It\'s his ultimate act of public indecency and attention-seeking.\n    *   **"Ooh la la!":** While not his primary catchphrase, Shin-chan often 